<a href="https://colab.research.google.com/github/cbonnin88/EcoVolt-HR/blob/main/Compensation_Decision_Support_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip install streamlit polars plotly -q
!pip install pyngrok -q

In [23]:
%%writefile app.py
import streamlit as st
import polars as pl
import plotly.express as px
import pandas as pd
import gdown as gd

# 1. Setup & Logo
st.set_page_config(page_title='EcoVolt Compensation Tool', layout='wide')
st.title("⚡ EcoVolt Renewables: Compensation Decision Tool")
st.markdown("---")
st.divider()

# 2. Data Loading with Google Drive Integration
@st.cache_data
def load_data():
    url_employees = 'https://drive.google.com/uc?id=10GJ4H7tVZpFu_hM6cegvpqUrR7bSXSg2'
    url_compensation = 'https://drive.google.com/uc?id=1rZJgQSXrh68DX6cdm0NGRJhk_5xZumTT'

    # Download files from Google Drive
    gd.download(url_employees, 'employees.csv', quiet=True)
    gd.download(url_compensation, 'compensation.csv', quiet=True)

    try:
        ee = pl.read_csv("employees.csv")
        comp = pl.read_csv("compensation.csv")

        # Join and ensure no nulls in critical math columns
        df = ee.join(comp, on="Employee_ID").with_columns([
            pl.col("Annual_Base_EUR").fill_null(0),
            pl.col("Market_Midpoint_EUR").fill_null(1)
        ])

        # Add Compa-Ratio column
        df = df.with_columns(
            (pl.col("Annual_Base_EUR") / pl.col("Market_Midpoint_EUR")).alias("Compa_Ratio")
        )
        return df
    except Exception as e:
        st.error(f"Error processing data: {e}")
        return None

df = load_data()

# 3. Main Application Logic
if df is not None:
    st.sidebar.header("Filter Analytics")

    # Get sorted unique lists for dropdowns using Polars methods
    depts = df["Department"].unique().sort().to_list()
    levels = df["Job_Level"].unique().sort().to_list()

    target_dept = st.sidebar.selectbox("Select Department", depts)
    target_level = st.sidebar.selectbox("Select Job Level", levels)

    # 4. Filter Data and Check for Empty Results
    filtered = df.filter(
        (pl.col("Department") == target_dept) &
        (pl.col("Job_Level") == target_level)
    )

    if len(filtered) > 0:
        # 5. KPI Metrics Row
        col1, col2, col3 = st.columns(3)

        # Calculate means safely (Returns None if empty, handled by the len() check above)
        avg_pay = filtered["Annual_Base_EUR"].mean()
        avg_compa = filtered["Compa_Ratio"].mean()

        with col1:
            st.metric("Headcount", len(filtered))
        with col2:
            st.metric("Avg Base Salary", f"€{avg_pay:,.0f}")
        with col3:
            # Show visual delta against market midpoint (1.0)
            delta_val = round(avg_compa - 1.0, 2)
            st.metric("Avg Compa-Ratio", f"{avg_compa:.2f}", delta=f"{delta_val} vs Market")

        # 6. Visualizations
        st.subheader(f"Insights for {target_dept} ({target_level})")

        c1, c2 = st.columns(2)
        with c1:
            fig_hist = px.histogram(filtered.to_pandas(), x="Annual_Base_EUR",
                                   title="Salary Distribution",
                                   color_discrete_sequence=["#00CC96"],
                                   labels={'Annual_Base_EUR': 'Base Salary (EUR)'})
            st.plotly_chart(fig_hist, use_container_width=True)

        with c2:
            fig_box = px.box(filtered.to_pandas(), x="Remote_Type", y="Annual_Base_EUR",
                            title="Pay Spread by Work Mode",
                            color="Remote_Type",
                            color_discrete_sequence=px.colors.qualitative.Pastel)
            st.plotly_chart(fig_box, use_container_width=True)

        # 7. Fair Pay Recommendation Tool
        st.divider()
        st.header("🔍 New Hire Fairness Calculator")

        exp_col, rem_col, calc_col = st.columns(3)
        with exp_col:
            exp = st.number_input("Relevant Experience (Years)", 0, 40, 5)
        with rem_col:
            rem_choice = st.selectbox("Candidate Work Mode", ["Office", "Hybrid", "Remote"])
        with calc_col:
            st.write(" ") # Vertical alignment spacer
            if st.button("Calculate Recommended Salary"):
                # Simulation of ML logic: Peer Avg + Tenure Premium
                tenure_premium = exp * 1500
                rec_salary = avg_pay + (tenure_premium if exp > 2 else 0)
                st.success(f"Recommended Offer: **€{rec_salary:,.2f}**")

    else:
        # Handle cases where the filter combination returns no rows
        st.warning(f"⚠️ No employee data available for **{target_dept}** at level **{target_level}**.")
        st.info("Try selecting a different Department or Job Level in the sidebar.")
else:
    st.error("Application failed to initialize. Please check your data sources.")



Overwriting app.py


In [22]:
!pkill streamlit

In [24]:
from pyngrok import ngrok
from google.colab import userdata

NGROK_AUTH_TOKEN_KEY = 'token'
ngrok_token = userdata.get(NGROK_AUTH_TOKEN_KEY)

if ngrok_token:
    ngrok.set_auth_token(ngrok_token)
else:
    print(f"Warning: ngrok auth token not found in Colab secrets. Please add it using the key '{NGROK_AUTH_TOKEN_KEY}'.")
    print("You can get your token from https://dashboard.ngrok.com/auth/your-authtoken")
    print("Without the token, ngrok will not be able to create a tunnel.")

# Connect to Streamlit port
public_url = ngrok.connect(8501)
print(f"Click here to open your App: {public_url}")

# Run Streamlit
!streamlit run app.py --server.port 8501

Click here to open your App: NgrokTunnel: "https://151b-34-53-65-124.ngrok-free.app" -> "http://localhost:8501"



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.53.65.124:8501

2026-04-02 08:17:16.828 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-04-02 08:17:16.884 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-04-02 08:17:22.630 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use 